In [15]:
import os, sys
from pathlib import Path

def add_project_path(module_folder="model_implementation"):
    candidates = [
        os.path.abspath("."),
        os.path.abspath("../src"),
        os.path.abspath("src"),
    ]

    for path in candidates:
        if os.path.exists(os.path.join(path, module_folder)):
            if path not in sys.path:
                sys.path.append(path)
            return path

    raise ImportError(f"Could not find '{module_folder}' in current or parent directory")

SRC_PATH = add_project_path("model_implementation")
add_project_path("cnn")
add_project_path("rnn")
ROOT = Path(SRC_PATH).parent.resolve()
print("ROOT:", ROOT)

ROOT: /content


In [16]:
from pathlib import Path
from datetime import datetime
import gc
import itertools
import json
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    import tensorflow as tf
    from tensorflow.keras import layers, models
except Exception as exc:
    tf = None
    layers = models = None
    print("TensorFlow belum tersedia:", exc)

try:
    from sklearn.metrics import f1_score
    from sklearn.model_selection import train_test_split
except Exception as exc:
    f1_score = None
    train_test_split = None
    print("scikit-learn belum tersedia:", exc)

from cnn.layers import Conv2D, LocallyConnected2D, MaxPooling2D, AveragePooling2D, GlobalMaxPooling2D, GlobalAveragePooling2D, Flatten, Dense, Sequential
from cnn.utility import image_loader, batch_loader, image_paths, feature_extractor
from common.io import save_json, load_json, load_npy, save_npy
from rnn.preprocess import read_captions, prep_data, load_vocab
from rnn.sequences import align_features_to_captions, teacher_pairs
from rnn.keras_models import build_preinject, compile_model
from rnn.train import grid_cfg
from rnn.weights import export_weights
from rnn.decode import decode_batch
from rnn.caption_decoder import build_decoder
from rnn.evaluate import eval_keras as eval_caption_keras, eval_caption_decoder, hist_sum, seq_tokens
from common.metrics import bleu4_score, meteor_safe

In [21]:
SEED = 42
IMAGE_SIZE = (150, 150)
BATCH_SIZE = 32
EPOCHS = 10
MAX_CAPTION_LENGTH = 34

np.random.seed(SEED)
plt.style.use("seaborn-v0_8")

DATA_DIR = ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
FEATURE_DIR = DATA_DIR / "features"
VOCAB_DIR = DATA_DIR / "vocab"
MODEL_DIR = ROOT / "models"
CNN_MODEL_DIR = MODEL_DIR / "cnn"
RNN_MODEL_DIR = MODEL_DIR / "rnn"
REPORT_DIR = ROOT / "reports"
TABLE_DIR = REPORT_DIR / "tables"
FIG_DIR = REPORT_DIR / "figures"

for path in [FEATURE_DIR, VOCAB_DIR, CNN_MODEL_DIR, RNN_MODEL_DIR, TABLE_DIR, FIG_DIR]:
    path.mkdir(parents=True, exist_ok=True)

CATEGORIES = {
    "buildings": 0,
    "forest": 1,
    "glacier": 2,
    "mountain": 3,
    "sea": 4,
    "street": 5,
}
INV_CATEGORIES = {v: k for k, v in CATEGORIES.items()}

if tf is not None:
    gpu_devices = tf.config.list_physical_devices("GPU")
    if gpu_devices:
        print("GPU Detected:", gpu_devices)
        for gpu in gpu_devices:
            tf.config.experimental.set_memory_growth(gpu, True)
    else:
        print("No GPU found, defaulting to CPU.")

GPU Detected: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [18]:
# !pip install -q kaggle

# from google.colab import files
# files.upload()  # upload kaggle.json

# !mkdir -p ~/.kaggle
# !cp kaggle.json ~/.kaggle/
# !chmod 600 ~/.kaggle/kaggle.json

# !kaggle datasets download -d puneet6060/intel-image-classification -p /content/
# !unzip -q /content/intel-image-classification.zip -d /content/intel-image-classification

# print("Dataset downloaded and extracted.")

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/puneet6060/intel-image-classification
License(s): copyright-authors
100% 346M/346M [00:01<00:00, 206MB/s]

Dataset downloaded and extracted.


In [22]:
# from pathlib import Path

# ROOT_DIR = Path("/content/intel-image-classification")

# image_extensions = {
#     ".jpg",
#     ".jpeg",
#     ".png",
#     ".bmp",
#     ".webp"
# }

# total_images = 0

# for folder in ROOT_DIR.rglob("*"):
#     if folder.is_dir():
#         count = sum(
#             1
#             for file in folder.iterdir()
#             if file.is_file() and file.suffix.lower() in image_extensions
#         )

#         if count > 0:
#             print(f"{folder}: {count}")
#             total_images += count

# print(f"\nTotal images: {total_images}")

/content/intel-image-classification/seg_pred/seg_pred: 7301
/content/intel-image-classification/seg_train/seg_train/mountain: 2512
/content/intel-image-classification/seg_train/seg_train/glacier: 2404
/content/intel-image-classification/seg_train/seg_train/buildings: 2191
/content/intel-image-classification/seg_train/seg_train/forest: 2271
/content/intel-image-classification/seg_train/seg_train/sea: 2274
/content/intel-image-classification/seg_train/seg_train/street: 2382
/content/intel-image-classification/seg_test/seg_test/mountain: 525
/content/intel-image-classification/seg_test/seg_test/glacier: 553
/content/intel-image-classification/seg_test/seg_test/buildings: 437
/content/intel-image-classification/seg_test/seg_test/forest: 474
/content/intel-image-classification/seg_test/seg_test/sea: 510
/content/intel-image-classification/seg_test/seg_test/street: 501

Total images: 24335


In [20]:
from pathlib import Path
import numpy as np
from sklearn.model_selection import train_test_split

DATASET_DIR = Path("/content/intel-image-classification")

TRAIN_DIR = DATASET_DIR / "seg_train" / "seg_train"
TEST_DIR  = DATASET_DIR / "seg_test" / "seg_test"
PRED_DIR  = DATASET_DIR / "seg_pred" / "seg_pred"

CATEGORIES = {
    "buildings": 0,
    "forest": 1,
    "glacier": 2,
    "mountain": 3,
    "sea": 4,
    "street": 5,
}

def image_paths(folder):
    return sorted(Path(folder).glob("*.jpg"))

def load_intel_dataset(root_path, target_size=(150, 150)):
    X, y = [], []
    root = Path(root_path)

    for category, label in CATEGORIES.items():
        cat_path = root / category
        if not cat_path.exists():
            print(f"Folder tidak ditemukan: {cat_path}")
            continue

        print(f"Loading {category}...")
        for image_path in image_paths(cat_path):
            try:
                img = tf.keras.utils.load_img(image_path, target_size=target_size)
                img = tf.keras.utils.img_to_array(img) / 255.0
                X.append(img)
                y.append(label)
            except Exception as exc:
                print(f"Error loading {image_path}: {exc}")

    return np.asarray(X, dtype="float32"), np.asarray(y, dtype="int32")

print("TRAIN_DIR:", TRAIN_DIR)
print("TEST_DIR :", TEST_DIR)

X_all, y_all = load_intel_dataset(TRAIN_DIR, target_size=IMAGE_SIZE)
X_test, y_test = load_intel_dataset(TEST_DIR, target_size=IMAGE_SIZE)

X_train, X_val, y_train, y_val = train_test_split(
    X_all,
    y_all,
    test_size=0.2,
    random_state=SEED,
    stratify=y_all
)

print("Train:", X_train.shape, y_train.shape)
print("Val  :", X_val.shape, y_val.shape)
print("Test :", X_test.shape, y_test.shape)

TRAIN_DIR: /content/intel-image-classification/seg_train/seg_train
TEST_DIR : /content/intel-image-classification/seg_test/seg_test
Loading buildings...
Loading forest...
Loading glacier...
Loading mountain...
Loading sea...
Loading street...
Loading buildings...
Loading forest...
Loading glacier...
Loading mountain...
Loading sea...
Loading street...
Train: (11227, 150, 150, 3) (11227,)
Val  : (2807, 150, 150, 3) (2807,)
Test : (3000, 150, 150, 3) (3000,)


In [ ]:
def prepare_dataset_safe(X, y, batch_size=32, shuffle=False):
    def generator():
        for i in range(len(X)):
            yield X[i], y[i]
    dataset = tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            tf.TensorSpec(shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3), dtype=tf.float32),
            tf.TensorSpec(shape=(), dtype=tf.int64),
        ),
    )
    if shuffle:
        dataset = dataset.shuffle(buffer_size=min(500, len(X)))
    return dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)

class LocalConv2D(tf.keras.layers.Layer if tf is not None else object):
    def __init__(self, filters, kernel_size, strides=(1, 1), activation=None, **kwargs):
        super().__init__(**kwargs)
        self.filters = int(filters)
        self.kernel_size = kernel_size if isinstance(kernel_size, tuple) else (int(kernel_size), int(kernel_size))
        self.strides = strides if isinstance(strides, tuple) else (int(strides), int(strides))
        self.activation_name = activation
        self.activation = tf.keras.activations.get(activation) if tf is not None else None

    def build(self, input_shape):
        channels = int(input_shape[-1])
        height, width = int(input_shape[1]), int(input_shape[2])
        k_h, k_w = self.kernel_size
        s_h, s_w = self.strides
        self.out_h = (height - k_h) // s_h + 1
        self.out_w = (width - k_w) // s_w + 1
        self.kernel = self.add_weight(name='kernel', shape=(self.out_h * self.out_w, k_h * k_w * channels, self.filters), initializer='glorot_uniform', trainable=True)
        self.bias = self.add_weight(name='bias', shape=(self.out_h * self.out_w, self.filters), initializer='zeros', trainable=True)

    def call(self, inputs):
        k_h, k_w = self.kernel_size
        s_h, s_w = self.strides
        channels = int(inputs.shape[-1])
        patches = tf.image.extract_patches(inputs, sizes=[1, k_h, k_w, 1], strides=[1, s_h, s_w, 1], rates=[1, 1, 1, 1], padding='VALID')
        batch = tf.shape(inputs)[0]
        patches = tf.reshape(patches, (batch, self.out_h * self.out_w, k_h * k_w * channels))
        output = tf.einsum('bpc,pco->bpo', patches, self.kernel) + self.bias
        output = tf.reshape(output, (batch, self.out_h, self.out_w, self.filters))
        if self.activation is not None:
            output = self.activation(output)
        return output

    def get_config(self):
        config = super().get_config()
        config.update({'filters': self.filters, 'kernel_size': self.kernel_size, 'strides': self.strides, 'activation': self.activation_name})
        return config

if tf is not None:
    tf.keras.utils.get_custom_objects()['LocalConv2D'] = LocalConv2D

def build_keras_cnn(config, local=False):
    model = models.Sequential(name=f"{'Local' if local else 'CNN'}_l{config['num_layers']}_f{config['filters']}_k{config['kernel_size']}_{config['pooling_type']}")
    conv_layer = LocalConv2D if local else layers.Conv2D
    model.add(conv_layer(config['filters'], (config['kernel_size'], config['kernel_size']), activation='relu', input_shape=(*IMAGE_SIZE, 3)))
    for _ in range(config['num_layers'] - 1):
        model.add(conv_layer(config['filters'], (config['kernel_size'], config['kernel_size']), activation='relu'))
        if config['pooling_type'] == 'max':
            model.add(layers.MaxPooling2D((2, 2)))
        else:
            model.add(layers.AveragePooling2D((2, 2)))
    model.add(layers.Flatten())
    model.add(layers.Dense(128, activation='relu', name='dense_hidden'))
    model.add(layers.Dense(len(CATEGORIES), activation='softmax', name='class_output'))
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

cnn_results_registry = []
cnn_histories = []
local_cnn_record = None

if tf is not None and X_train is not None:
    train_ds = prepare_dataset_safe(X_train, y_train, batch_size=BATCH_SIZE, shuffle=True)
    val_ds = prepare_dataset_safe(X_val, y_val, batch_size=BATCH_SIZE)

    keras_variations = {
        'num_layers': [2, 3],
        'filters': [32, 64],
        'kernel_size': [3, 5],
        'pooling_type': ['max', 'avg'],
    }
    keys, values = zip(*keras_variations.items())
    configs = [dict(zip(keys, item)) for item in itertools.product(*values)]

    for idx, cfg in enumerate(configs):
        if idx < 7:
          continue
        print(f"\n--- Running CNN Experiment {idx + 1}/16: {cfg} ---")
        tf.keras.backend.clear_session()
        model = build_keras_cnn(cfg, local=False)
        history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, verbose=1)
        y_pred = np.argmax(model.predict(X_test, batch_size=BATCH_SIZE, verbose=0), axis=1) if X_test is not None else []
        macro_f1 = float(f1_score(y_test, y_pred, average='macro')) if f1_score is not None and X_test is not None else None
        model_path = CNN_MODEL_DIR / f"cnn_exp_{idx + 1:02d}.keras"
        model.save(model_path)
        record = {'index': idx + 1, **cfg, 'model_path': str(model_path), 'macro_f1': macro_f1, 'shared_parameters': True}
        cnn_results_registry.append(record)
        cnn_histories.append({'index': idx + 1, 'config': cfg, 'history': hist_sum(history)})
        gc.collect()

    best_cfg = max(cnn_results_registry, key=lambda item: item.get('macro_f1') or 0.0)
    local_cfg = {key: best_cfg[key] for key in ['num_layers', 'filters', 'kernel_size', 'pooling_type']}
    print("\n--- Running non-shared LocalConv2D variant:", local_cfg, "---")
    tf.keras.backend.clear_session()
    local_model = build_keras_cnn(local_cfg, local=True)
    local_history = local_model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, verbose=1)
    local_pred = np.argmax(local_model.predict(X_test, batch_size=BATCH_SIZE, verbose=0), axis=1) if X_test is not None else []
    local_f1 = float(f1_score(y_test, local_pred, average='macro')) if f1_score is not None and X_test is not None else None
    local_path = CNN_MODEL_DIR / 'cnn_local_best_config.keras'
    local_model.save(local_path)
    local_cnn_record = {'config': local_cfg, 'model_path': str(local_path), 'macro_f1': local_f1, 'shared_parameters': False}

    save_json(cnn_results_registry, TABLE_DIR / 'cnn_records.json')
    save_json(cnn_histories, TABLE_DIR / 'cnn_histories.json')
    save_json(local_cnn_record, TABLE_DIR / 'cnn_local_record.json')

    cnn_df = pd.DataFrame(cnn_results_registry)
    cnn_df.to_csv(TABLE_DIR / 'cnn_records.csv', index=False)
    if not cnn_df.empty and 'macro_f1' in cnn_df:
        factor_tables = {
            'cnn_by_num_layers.csv': cnn_df.groupby('num_layers', as_index=False)['macro_f1'].mean(),
            'cnn_by_filters.csv': cnn_df.groupby('filters', as_index=False)['macro_f1'].mean(),
            'cnn_by_kernel_size.csv': cnn_df.groupby('kernel_size', as_index=False)['macro_f1'].mean(),
            'cnn_by_pooling_type.csv': cnn_df.groupby('pooling_type', as_index=False)['macro_f1'].mean(),
        }
        for filename, table in factor_tables.items():
            table.to_csv(TABLE_DIR / filename, index=False)
        plt.figure(figsize=(10, 4))
        plt.bar(cnn_df['index'].astype(str), cnn_df['macro_f1'].fillna(0.0))
        plt.xlabel('CNN experiment')
        plt.ylabel('Macro F1')
        plt.tight_layout()
        plt.savefig(FIG_DIR / 'cnn_macro_f1.png', dpi=160)
        plt.show()
else:
    print("CNN training dilewati.")

pd.DataFrame(cnn_results_registry)


--- Running CNN Experiment 8/16: {'num_layers': 2, 'filters': 64, 'kernel_size': 5, 'pooling_type': 'avg'} ---
Epoch 1/10


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


    351/Unknown 53s 115ms/step - accuracy: 0.3976 - loss: 2.6816

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


351/351 ━━━━━━━━━━━━━━━━━━━━ 63s 142ms/step - accuracy: 0.4874 - loss: 1.5408 - val_accuracy: 0.5750 - val_loss: 1.1395
Epoch 2/10
351/351 ━━━━━━━━━━━━━━━━━━━━ 34s 97ms/step - accuracy: 0.6467 - loss: 0.9269 - val_accuracy: 0.5757 - val_loss: 1.0938
Epoch 3/10
351/351 ━━━━━━━━━━━━━━━━━━━━ 34s 95ms/step - accuracy: 0.7757 - loss: 0.6036 - val_accuracy: 0.6698 - val_loss: 0.9342
Epoch 4/10
351/351 ━━━━━━━━━━━━━━━━━━━━ 34s 95ms/step - accuracy: 0.8807 - loss: 0.3426 - val_accuracy: 0.6680 - val_loss: 1.2481
Epoch 5/10
351/351 ━━━━━━━━━━━━━━━━━━━━ 34s 96ms/step - accuracy: 0.9439 - loss: 0.1761 - val_accuracy: 0.6765 - val_loss: 1.6344
Epoch 6/10
351/351 ━━━━━━━━━━━━━━━━━━━━ 34s 96ms/step - accuracy: 0.9667 - loss: 0.1108 - val_accuracy: 0.6733 - val_loss: 1.7277
Epoch 7/10
351/351 ━━━━━━━━━━━━━━━━━━━━ 34s 95ms/step - accuracy: 0.9832 - loss: 0.0579 - val_accuracy: 0.6801 - val_loss: 2.1089
Epoch 8/10
351/351 ━━━━━━━━━━━━━━━━━━━━ 34s 96ms/step - accuracy: 0.9810 - loss: 0.0719 - val_accura

In [ ]:
def keras_to_scratch(keras_model):
    scratch_layers = []
    for layer in keras_model.layers:
        name = layer.__class__.__name__
        weights = layer.get_weights()
        activation = getattr(getattr(layer, 'activation', None), '__name__', None)
        if name == 'Conv2D':
            conv = Conv2D(filters=layer.filters, kernel_size=layer.kernel_size, strides=layer.strides, padding=0, activation=activation)
            conv.load_keras(weights)
            scratch_layers.append(conv)
        elif name == 'MaxPooling2D':
            scratch_layers.append(MaxPooling2D(pool_size=layer.pool_size, strides=layer.strides))
        elif name == 'AveragePooling2D':
            scratch_layers.append(AveragePooling2D(pool_size=layer.pool_size, strides=layer.strides))
        elif name == 'Flatten':
            scratch_layers.append(Flatten())
        elif name == 'Dense':
            dense = Dense(units=layer.units, activation=activation)
            dense.load_keras(weights)
            scratch_layers.append(dense)
    return Sequential(scratch_layers)

cnn_manual_comparison = []

if cnn_results_registry and X_test is not None and tf is not None:
    best = max(cnn_results_registry, key=lambda item: item.get('macro_f1') or 0.0)
    best_model = tf.keras.models.load_model(best['model_path'])
    sample_x = X_test[: min(64, len(X_test))]
    sample_y = y_test[: len(sample_x)]
    scratch_model = keras_to_scratch(best_model)
    keras_pred = np.argmax(best_model.predict(sample_x, verbose=0), axis=1)
    scratch_pred = np.argmax(scratch_model.predict(sample_x), axis=1)
    cnn_manual_comparison = [
        {'implementation': 'keras_shared', 'macro_f1': float(f1_score(sample_y, keras_pred, average='macro')), 'params': int(best_model.count_params())},
        {'implementation': 'scratch_numpy_shared', 'macro_f1': float(f1_score(sample_y, scratch_pred, average='macro')), 'params': scratch_model.count_params()},
    ]
    if local_cnn_record is not None:
        cnn_manual_comparison.append({'implementation': 'keras_non_shared_local', 'macro_f1': local_cnn_record['macro_f1'], 'params': int(local_model.count_params())})
    save_json(cnn_manual_comparison, TABLE_DIR / 'cnn_manual_comparison.json')
    pd.DataFrame(cnn_manual_comparison).to_csv(TABLE_DIR / 'cnn_manual_comparison.csv', index=False)
else:
    print("CNN scratch comparison dilewati.")

cnn_manual_comparison